# 第 3 章：编码注意力机制

演进：简单自注意力(无权重) → Q/K/V → 因果掩码 → 多头。核心 MultiHeadAttention 在 src/gpt/attention.py。

In [ ]:
import torch, torch.nn as nn
inputs = torch.tensor([[0.43,0.15,0.89],[0.55,0.87,0.66],[0.57,0.85,0.64],[0.22,0.58,0.33],[0.77,0.25,0.10],[0.05,0.80,0.55]])
print("输入:", inputs.shape)

## 1. 简单自注意力（无权重）
点积→softmax→加权求和

In [ ]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
print("上下文向量:", (attn_weights @ inputs).shape)

## 2. 带 Q/K/V 可训练权重
注意力 = softmax(Q·Kᵀ/√d_k)·V

In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(nn.init.xavier_uniform_(torch.empty(d_in, d_out)))
        self.W_key = nn.Parameter(nn.init.xavier_uniform_(torch.empty(d_in, d_out)))
        self.W_value = nn.Parameter(nn.init.xavier_uniform_(torch.empty(d_in, d_out)))
    def forward(self, x):
        q,k,v = x@self.W_query, x@self.W_key, x@self.W_value
        return torch.softmax(q@k.T/k.shape[-1]**0.5, dim=-1) @ v
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query, self.W_key, self.W_value = (nn.Linear(d_in,d_out,bias=qkv_bias) for _ in range(3))
    def forward(self, x):
        k,q,v = self.W_key(x), self.W_query(x), self.W_value(x)
        return torch.softmax(q@k.T/k.shape[-1]**0.5, dim=-1) @ v
torch.manual_seed(123)
print("v1:", SelfAttention_v1(3,2)(inputs).shape, "v2:", SelfAttention_v2(3,2)(inputs).shape)

## 3. 因果注意力（掩码+dropout+批处理）
上三角置 -∞

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query, self.W_key, self.W_value = (nn.Linear(d_in,d_out,bias=qkv_bias) for _ in range(3))
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())
    def forward(self, x):
        b,n,_ = x.shape
        k,q,v = self.W_key(x), self.W_query(x), self.W_value(x)
        s = q@k.transpose(1,2); s.masked_fill_(self.mask.bool()[:n,:n], -torch.inf)
        return self.dropout(torch.softmax(s/k.shape[-1]**0.5, dim=-1)) @ v
batch = torch.stack((inputs,inputs),dim=0)
print("因果:", CausalAttention(3,2,6,0.0)(batch).shape)

## 4. 多头注意力（src/gpt 复用）

In [ ]:
import sys; sys.path.insert(0,"..")
from src.gpt import MultiHeadAttention
mha = MultiHeadAttention(d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2)
print("多头:", mha(batch).shape)